In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, LeakyReLU, Input
from tensorflow.keras.optimizers import Adam
import os

# --- Configuration ---
SEQ_LEN = 60       # The number of days in a sequence
N_FEATURES = 1     # We are only using 'Log Return'
LATENT_DIM = 100   # Size of the random noise vector feeding the Generator

# Define Paths
BASE_PATH = os.path.dirname(os.getcwd()) 
MODEL_PATH = os.path.join(BASE_PATH, 'models')
os.makedirs(MODEL_PATH, exist_ok=True)

# --- 1. Build the Generator ---
# Objective: Input Random Noise 
def build_generator():
    model = Sequential(name="Generator")
    
    # We use a Dense layer first to upscale the noise to enough neurons 
    # to reshape it into a sequence. 
    # We want 60 time steps * 1 features = 60 neurons, but we scale it up for complexity.
    model.add(Input(shape=(LATENT_DIM,)))
    model.add(Dense(SEQ_LEN * 128)) 
    model.add(LeakyReLU(alpha=0.2))
    
    # Reshape into 3D for LSTM: (Samples, Time Steps, Features)
    model.add(tf.keras.layers.Reshape((SEQ_LEN, 128)))
    
    # LSTM Layers to learn time-series patterns
    # return_sequences=True is needed if we stack LSTMs
    model.add(LSTM(128, return_sequences=True))
    model.add(LSTM(64, return_sequences=True))
    
    # Final Output Layer
    # We use 'tanh' because our data is normalized between -1 and 1
    model.add(Dense(N_FEATURES, activation='tanh'))
    
    return model

generator = build_generator()
print("\n--- Generator Summary ---")
generator.summary()

# --- 2. Build the Discriminator ---
# Objective: Input Sequence (60, 1) -> Output Probability (Real vs Fake)
def build_discriminator():
    model = Sequential(name="Discriminator")
    
    model.add(Input(shape=(SEQ_LEN, N_FEATURES)))
    
    # LSTM layers to analyze the sequence pattern
    model.add(LSTM(64, return_sequences=True))
    model.add(LeakyReLU(alpha=0.2))
    
    # Second LSTM layer does not return sequences, it condenses info into a single vector
    model.add(LSTM(32)) 
    model.add(LeakyReLU(alpha=0.2))
    
    # Output Layer: Single neuron with Sigmoid (0 = Fake, 1 = Real)
    model.add(Dense(1, activation='sigmoid'))
    
    # Compile the Discriminator immediately (Generator is compiled in the combined GAN)
    optimizer = Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    
    return model

discriminator = build_discriminator()
print("\n--- Discriminator Summary ---")
discriminator.summary()

# --- 3. Build the Combined GAN ---
# We stack Generator and Discriminator to train the Generator.
# When training the Generator, the Discriminator weights must be frozen.

discriminator.trainable = False # Freeze D

gan_input = Input(shape=(LATENT_DIM,))
generated_seq = generator(gan_input)
gan_output = discriminator(generated_seq)

gan = tf.keras.models.Model(gan_input, gan_output, name="GAN_Combined")

gan_optimizer = Adam(learning_rate=0.0002, beta_1=0.5)
gan.compile(loss='binary_crossentropy', optimizer=gan_optimizer)

print("\n--- Combined GAN Summary ---")
gan.summary()

print("\nStep 2 Complete: Models built successfully.")

C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\Yahya\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 


--- Generator Summary ---


c:\Users\Yahya\anaconda3\envs\myenv\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 7680)           │       775,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 7680)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 60, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 60, 1)          │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Generator (Sequential)          │ (None, 60, 1)          │       956,737 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Discriminator (Sequential)      │ (None, 1)              │        29,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 986,082 (3.76 MB)

 Trainable params: 956,737 (3.65 MB)

 Non-trainable params: 29,345 (114.63 KB)


Step 2 Complete: Models built successfully.
